# Day 077 — Solution: Real-Time Vision Agent

In [ ]:
_SRC = '"""live_vision.py — Day 077: Real-Time Vision Agent.\n\nCapture frames from a webcam or device, analyze them with a vision LLM,\nand save results. All three capabilities are injectable for headless testing.\n\nFunctions:\n    open_camera        — open a camera device as a VideoCapture object\n    read_frame         — read one frame from an open capture\n    frame_to_image     — convert BGR ndarray to PIL Image (RGB)\n    analyze_frame      — vision LLM analysis of one frame\n    save_frame         — write a frame to disk as PNG\n    should_analyze     — rate-control predicate (every_n frames)\n    capture_frames     — collect n_frames from camera into a list\n    analyze_stream     — capture + analyze n_frames, return results\n    LiveVisionAgent    — context-manager camera agent with vision\n\nSetup:\n    pip install pillow ollama opencv-python-headless\n    ollama pull llava\n"""\nimport io\nimport base64\nfrom pathlib import Path\n\n\ndef open_camera(device=0, camera_fn=None):\n    """Open a camera device and return a VideoCapture-compatible object.\n\n    Args:\n        device:    integer device index (0 = default webcam) or file path\n        camera_fn: callable(device) -> cap for testing\n    Returns:\n        VideoCapture object with .isOpened(), .read(), .release()\n    Raises:\n        RuntimeError if the camera cannot be opened\n    """\n    if camera_fn is not None:\n        return camera_fn(device)\n    import cv2\n    cap = cv2.VideoCapture(device)\n    if not cap.isOpened():\n        raise RuntimeError(f\'Cannot open camera device {device}\')\n    return cap\n\n\ndef read_frame(cap):\n    """Read one frame from an open capture.\n\n    Args:\n        cap: VideoCapture or compatible object\n    Returns:\n        (success: bool, frame: ndarray | None)\n    """\n    return cap.read()\n\n\ndef frame_to_image(frame):\n    """Convert a BGR ndarray frame to a PIL Image in RGB mode.\n\n    Args:\n        frame: ndarray shape (H, W, 3) in BGR channel order (OpenCV convention)\n    Returns:\n        PIL Image in RGB mode\n    """\n    from PIL import Image\n    rgb = frame[:, :, ::-1]\n    return Image.fromarray(rgb)\n\n\ndef analyze_frame(frame, question, analyze_fn=None):\n    """Analyze a camera frame with a vision LLM.\n\n    Args:\n        frame:      BGR ndarray\n        question:   natural-language question about the frame\n        analyze_fn: callable(pil_image, question) -> str for testing\n    Returns:\n        model answer string\n    """\n    image = frame_to_image(frame)\n    if analyze_fn is not None:\n        return analyze_fn(image, question)\n    import ollama\n    buf = io.BytesIO()\n    image.save(buf, format=\'PNG\')\n    img_b64 = base64.b64encode(buf.getvalue()).decode()\n    resp = ollama.chat(\n        model=\'llava\',\n        messages=[{\'role\': \'user\', \'content\': question, \'images\': [img_b64]}],\n    )\n    return resp[\'message\'][\'content\']\n\n\ndef save_frame(frame, path):\n    """Save a BGR frame to disk as a PNG image.\n\n    Args:\n        frame: BGR ndarray\n        path:  output path (str or Path)\n    Returns:\n        Path of the saved file\n    """\n    out = Path(path)\n    frame_to_image(frame).save(out, format=\'PNG\')\n    return out\n\n\ndef should_analyze(frame_count, every_n):\n    """Return True if this frame should be analyzed.\n\n    Args:\n        frame_count: zero-based frame counter\n        every_n:     analyze every Nth frame (1 = every frame)\n    Returns:\n        bool\n    """\n    return frame_count % every_n == 0\n\n\ndef capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None):\n    """Capture n_frames frames from a camera, rate-controlled by every_n.\n\n    Args:\n        device:    camera device index\n        n_frames:  how many frames to collect\n        every_n:   collect every Nth frame read from the camera\n        camera_fn: callable(device) -> cap for testing\n    Returns:\n        list of BGR ndarray frames\n    """\n    cap = open_camera(device=device, camera_fn=camera_fn)\n    frames = []\n    frame_count = 0\n    try:\n        while len(frames) < n_frames:\n            ret, frame = read_frame(cap)\n            if not ret:\n                break\n            if should_analyze(frame_count, every_n):\n                frames.append(frame)\n            frame_count += 1\n    finally:\n        cap.release()\n    return frames\n\n\ndef analyze_stream(device=0, task=\'Describe what you see.\', n_frames=5,\n                   every_n=1, camera_fn=None, analyze_fn=None):\n    """Capture frames and analyze each with a vision LLM.\n\n    Args:\n        device:     camera device index\n        task:       question or instruction for the vision LLM\n        n_frames:   number of frames to analyze\n        every_n:    analyze every Nth frame (skip the rest)\n        camera_fn:  callable(device) -> cap for testing\n        analyze_fn: callable(pil_image, question) -> str for testing\n    Returns:\n        list of dicts: [{frame_idx: int, description: str}, ...]\n    """\n    cap = open_camera(device=device, camera_fn=camera_fn)\n    results = []\n    frame_count = 0\n    analyzed = 0\n    try:\n        while analyzed < n_frames:\n            ret, frame = read_frame(cap)\n            if not ret:\n                break\n            if should_analyze(frame_count, every_n):\n                description = analyze_frame(frame, task, analyze_fn=analyze_fn)\n                results.append({\'frame_idx\': frame_count, \'description\': description})\n                analyzed += 1\n            frame_count += 1\n    finally:\n        cap.release()\n    return results\n\n\nclass LiveVisionAgent:\n    """Context-manager camera agent with vision analysis.\n\n    Opens a camera on enter, releases it on exit. Stores the last frame\n    captured so analyze/describe/save can be called without re-reading.\n\n    Example::\n\n        with LiveVisionAgent(\n            camera_fn=mock_camera,\n            analyze_fn=mock_analyze,\n        ) as agent:\n            frame = agent.read()\n            desc = agent.describe()\n            agent.save(frame, \'frame.png\')\n    """\n\n    def __init__(self, device=0, camera_fn=None, analyze_fn=None):\n        self._device = device\n        self._camera_fn = camera_fn\n        self._analyze_fn = analyze_fn\n        self._cap = None\n        self._last_frame = None\n\n    def open(self, device=None):\n        """Open the camera. Called automatically by __enter__."""\n        d = device if device is not None else self._device\n        self._cap = open_camera(device=d, camera_fn=self._camera_fn)\n        return self\n\n    def read(self):\n        """Read one frame; store as last frame. Returns ndarray or None."""\n        if self._cap is None:\n            raise RuntimeError(\'Camera not open: call open() first or use as context manager.\')\n        ret, frame = read_frame(self._cap)\n        if ret:\n            self._last_frame = frame\n        return frame if ret else None\n\n    def analyze(self, question, frame=None):\n        """Analyze a frame (or last captured frame) with a vision LLM."""\n        f = frame if frame is not None else self._last_frame\n        if f is None:\n            raise ValueError(\'No frame: call read() first or pass frame.\')\n        return analyze_frame(f, question, analyze_fn=self._analyze_fn)\n\n    def describe(self, frame=None):\n        """Describe what is visible in the frame."""\n        return self.analyze(\'Describe what you see in detail.\', frame=frame)\n\n    def save(self, path, frame=None):\n        """Save a frame (or last captured frame) to disk as PNG."""\n        f = frame if frame is not None else self._last_frame\n        if f is None:\n            raise ValueError(\'No frame: call read() first or pass frame.\')\n        return save_frame(f, path)\n\n    def close(self):\n        """Release the camera resource."""\n        if self._cap is not None:\n            self._cap.release()\n            self._cap = None\n\n    def __enter__(self):\n        self.open()\n        return self\n\n    def __exit__(self, *args):\n        self.close()\n'
from pathlib import Path
Path('live_vision.py').write_text(_SRC, encoding='utf-8')
print('live_vision.py written.')

In [ ]:

import numpy as np
from pathlib import Path
from live_vision import (
    open_camera, read_frame, frame_to_image, analyze_frame,
    save_frame, should_analyze, capture_frames, analyze_stream,
    LiveVisionAgent,
)
from PIL import Image as PILImage

def _make_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5):
        self._frames = [_make_frame() for _ in range(n)]
        self._idx = 0
    def isOpened(self): return True
    def read(self):
        if self._idx >= len(self._frames): return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self): pass
    def get(self, p): return 0.0

_mock_camera_fn = lambda d: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]

# 1. open_camera
cap = open_camera(device=0, camera_fn=_mock_camera_fn)
assert cap.isOpened()
print("✅ open_camera")

# 2. read_frame
ret, frame = read_frame(cap)
assert ret and isinstance(frame, np.ndarray)
print("✅ read_frame")

# 3. frame_to_image
img = frame_to_image(frame)
assert isinstance(img, PILImage.Image)
print("✅ frame_to_image")

# 4. analyze_frame
r = analyze_frame(frame, 'Q?', analyze_fn=_mock_analyze_fn)
assert isinstance(r, str)
print("✅ analyze_frame")

# 5. save_frame
import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    tmp = f.name
try:
    p = save_frame(frame, tmp)
    assert os.path.getsize(tmp) > 0
    print("✅ save_frame")
finally:
    os.unlink(tmp)

# 6. should_analyze
assert should_analyze(0, 5) and should_analyze(5, 5)
assert not should_analyze(1, 5) and not should_analyze(4, 5)
print("✅ should_analyze")

# 7. capture_frames
frames = capture_frames(n_frames=3, camera_fn=_mock_camera_fn)
assert len(frames) == 3 and all(isinstance(f, np.ndarray) for f in frames)
print("✅ capture_frames")

# 8. analyze_stream
results = analyze_stream(n_frames=2, camera_fn=_mock_camera_fn, analyze_fn=_mock_analyze_fn)
assert len(results) == 2 and all('frame_idx' in r and 'description' in r for r in results)
print("✅ analyze_stream")

# 9. LiveVisionAgent
with LiveVisionAgent(camera_fn=_mock_camera_fn, analyze_fn=_mock_analyze_fn) as agent:
    f = agent.read()
    assert isinstance(f, np.ndarray)
    d = agent.describe()
    assert isinstance(d, str)
assert agent._cap is None
print("✅ LiveVisionAgent (context manager)")

print("\nReal-Time Vision Agent complete!")
